# KI-FSPrompt Online-Inference

Trying KI-FSPrompt for single Text input. This notebook demonstrates how
ki-fsprompt operates. While the offline-inference version of KI-FSPrompt
proceeds through the stages, by processing a complete batch of texts in
every stage with vllm-Offline-Inference, loading LLMs into GPU-memory on demand,
this demo demonstrates the steps for a single input-text, using online-inference,
which requires that a LLM is served permanently on some API-Url. 

## Prerequisites

You need to start all three services used by ki-fsprompt, as specified in the
`docker-compose.yaml`.

1. Weaviate to serve the vector storage for `retrieve` and `map`
2. Huggingface-TEI to serve the embedding generation for `retrieve` and `map`
3. vLLM to serve a generative LLM for `complete` and `rank`

Also we assume that the weaviate collections for `retrieve` and `map` have been 
created using the respective dvc-stages in the `train` pipeline. If these do 
not exist, you can create them by running:

```bash
dvc repro -sf train/dvc.yaml:create_vocab_collection \
  train/dvc.yaml:create_train_collection
```

In [ ]:
# Only needed for debugging with VS Code
import debugpy
debugpy.listen(5678)
debugpy.wait_for_client() 

## Retrieval Stage

Im ersten Schritt wird die Vektordatenbank abgefragt, die die Texte aus dem
Train Datensatz, inklusive ihrer Schlagworte beinhaltet.

In [1]:
import pandas as pd
from pathlib import Path
from src.retrieve import ExampleRetriever   # Import fehlt – läuft nur, weil der Kernel ihn noch vom alten Lauf kennt

val = pd.read_csv("pipelines/preprocess/toc_strict/validate.csv")
input_text_data = val.iloc[[1]].reset_index(drop=True)   # reset_index, damit später [0] auch bei anderen Zeilen klappt

custom_instruction = Path("instructions/instruction_toc.txt").read_text()   # kein ResourceWarning mehr

retriever = ExampleRetriever(
    input_text_data=input_text_data,
    output_file=None,
    n_examples=5,
    collection_name="toc_strict_train",
    host="8090",
    debug=False,
)
retrieved_texts = retriever.retrieve_examples()

inp = input_text_data.iloc[0]
print(f"##### EINGABE  doc_id={inp.doc_id}")
print(inp.text[:600])
print("→ GOLD:", inp.label_texts, "\n")
print("#" * 60, "\n")

for _, r in retrieved_texts.iterrows():
    print(f"=== {r.prompt_doc_id}  distance={r.similarity:.3f}")
    print(r.prompt_text[:600])
    print("→", r.prompt_label_texts, "\n")


TypeError: argument of type 'method' is not a container or iterable

## Erzeuge einen Prompt aus dem neuen Text und den Beispielen

Wir benutzen die PromptBuilder Funktion, um aus den extrahierten Beispielen
`messages` für eine LLM-Anfrage zu konstruieren

In [ ]:
from src.PromptBuilder import IndividualPromptBuilder

prompt_builder = IndividualPromptBuilder(
    parsed_prompt_file=retrieved_texts,
    custom_instruction="Bitte extrahiere die Schlüsselwörter aus dem Text, wie in den Beispielen. Antworte im selben JSON-Format",
    debug=False,
)

messages = prompt_builder.build_messages(doc_id=str(input_text_data["doc_id"][0]), text=input_text_data["text"][0])
messages


[{'role': 'system',
  'content': 'Bitte extrahiere die Schlüsselwörter aus dem Text, wie in den Beispielen. Antworte im selben JSON-Format'},
 {'role': 'user',
  'content': 'Motivation und Zielstellung\nVorgehensweise\nWissensmanagement\nDer Wissensbegriff\nEpistemologische Betrachtung\nBetriebswirtschaftliche Betrachtung\nInformationstheoretische Betrachtung\nWissenstheoretische Betrachtung\nDer Begriff Wissensmanagement\nZiele und Aufgaben des Wissensmanagements\nWissensmanagement-Modelle\nHerausforderungen und Barrieren des Wissensmanagements\nTechnologische Grundlagen des Wissensmanagements\nWissensmanagementsysteme\nWissensrepräsentation und Inferenz\nLogikbasierte Wissensrepräsentation und Inferenz\nund -verarbeitung\nWissensmanagement im Kontext maschinellen Lernens\nZiele und Anforderungen des Wissensmanagements im Umfeld der industriellen Produktion\nGegebenheiten der industriellen Produktion\nBegriffsdefinition\nWissensmanagement im Umfeld der Produktion\nWissensmanagement na

## Führe den Completion-Schritt durch

Hier weichen wir vom Quellcode von KI-FSPrompt ab, weil das Skript `src/completion.py`
mit Batch-Processing und Offline-Inferenz arbeitet.

In [ ]:
LLM_API_URL = "http://localhost:9513/v1"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

from openai import OpenAI

client = OpenAI(
  base_url=LLM_API_URL, 
  api_key = "must be a non-empty string, but is not required for vLLM")

keyword_schema = {
    "type": "object",
    "properties": {
        "keywords": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 20,
        }
    },
    "required": ["keywords"],
    "additionalProperties": False,
}

response = client.chat.completions.create(
    model = MODEL_NAME,
    messages=messages,
    extra_body={
      "guided_json": keyword_schema,
      # chat_template not supported by mistral-7B-0p3
      #"chat_template_kwargs": {"enable_thinking": False}
      }
)

keywords = response.choices[0].message.content
print(keywords)

 {"keywords": ["Infrastrukturüberwachung", "Zustandsüberwachung", "Integrierende Systemarchitektur", "Netzinformationssysteme", "Verkehrstelematik", "Ontologien", "Beschreibungslogiken", "Sparql", "Topologiebezug", "Schienennetze", "Schichtige Architektur", "Funktionelle Schichten"]}


## Mapping auf GND Normdaten

Wir verwenden nun die weaviate-Collection, welche die GND mit Vektorisierung
zur Verfügung stellt, um die freien Schlagwörter auf mögliche GND-Einträge zu mappen.

In [ ]:
from src.mapping import Mapping
import json
import weaviate

db_connection = weaviate.connect_to_local(port=8087)

hyperparameters = {
  "host": '8090',
  "alpha": 0.7,
  "use_phrase": False,
  "search": "hybrid"
}

mapping = Mapping(
    hyperparameters=hyperparameters,
    collection_name="ki_fsprompt_vocab",
    phrase=None,
    debug=False,
    db_connection=db_connection,
)

keyword_list = json.loads(keywords)["keywords"]

results = []
for keyword in keyword_list:
    item = mapping.query_vector_database(candidate=keyword)
    for label, info in item.items():
        results.append({"keyword": keyword, **info})


mapped_keywords = pd.DataFrame(results)

db_connection.close()
mapped_keywords

,keyword,label_id,term,hybrid_score,cosine_similarity
0,Infrastrukturüberwachung,042839424,Verkehrsüberwachung,0.700000,0.849641
1,Zustandsüberwachung,943818761,Zustandsüberwachung,1.000000,0.999992
2,Integrierende Systemarchitektur,041393740,Systemarchitektur,0.782372,0.777381
3,Netzinformationssysteme,040728064,Informationssysteme,0.700000,0.860001
4,Verkehrstelematik,041878175,Verkehrstelematik,1.000000,0.999983
5,Ontologien,040756602,Ontologie,0.700000,0.798176
6,Beschreibungslogiken,943072654,Beschreibungslogik,0.700000,0.969375
7,Sparql,1000675467,SPARQL,0.815160,0.718483
8,Topologiebezug,04060425X,Topologie,0.700000,0.769912
9,Schienennetze,962072788,Schienenzeppelin,0.700000,0.721339
